# Creating the smooth twins

This is step 1 of the ISO end-to-end pipeline:

1. **this notebook** builds the smooth twins of the mocks,
2. [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
   between the probes,
3. [`analyse_datasets.ipynb`](analyse_datasets.ipynb) runs the joint fit,
4. [`generate_grid.ipynb`](generate_grid.ipynb) writes the full grid of runs.

A *smooth twin* reuses a mock's tracers, bandpower windows and covariance, but replaces its noisy
bandpowers with the theory at the fiducial of the configuration, so that the likelihood gives
chi-square = 0 when fit at that point. The twins are what the pipeline is validated on before it is
run on the mocks, and later on the real data.

Each twin is built by evaluating the likelihood on the **mock** files (`level="mock"`) at the
fiducial, and written where the `twin` section of `paths.yaml` expects it:

1. the galaxy x CMB-lensing twin, via `soliket.sacc_tools.smooth_twin_sacc`;
2. the MFLike and CMB-lensing twins, from one joint model, via
   `soliket.sacc_tools.smooth_mflike_sacc` and `smooth_twin_sacc`.

The fiducial is the one of the configuration (`ref.loc` of each parameter), so it must be the point
the mocks were generated at.

> **Configuration.** Every fiducial value, prior, modelling choice and file path comes from the
> configuration folder `$ISO_CONFIG` (by default [`example_config/`](example_config)), read through
> `iso_config.py`; the data files live under `$ISO_DATA` (by default the git-ignored `iso_data/`). See [`README.md`](README.md) for the setup. The data are used at
> three *levels*, sections of the configuration's `paths.yaml` that only differ by the files
> loaded: `mock` (realistic simulations), `twin` (their smooth twins) and `real` (the data).

In [ ]:
import numpy as np
import sacc

import iso_config

# Create the data folders listed in paths.yaml (the mocks go in theirs).
for folder in iso_config.make_data_dirs():
    print("folder:", folder)

MOCK = iso_config.paths("mock")
TWIN = iso_config.paths("twin")

> **First run:** the cell below installs the likelihoods' packages data (MFLike's data folder, and
> any file `paths.yaml` takes from `{packages}`, such as the mocks of `example_config/`). It is a
> no-op once the data is present.

In [ ]:
from cobaya.install import install
from cobaya.tools import resolve_packages_path

# Install only the likelihood data: CAMB comes from the Python environment (the
# `camb` package), so cobaya must not try to download and build its own copy.
# The two probes are installed separately, NOT via the multigaussian dataset: that
# one nests them in `MultiGaussianLikelihood.components`, where cobaya's installer
# cannot see them -- it reports success while downloading nothing.
for dataset in ["mflike", "lensing"]:
    likelihoods = iso_config.build_info(dataset, level="mock")["likelihood"]
    install({"likelihood": likelihoods}, path=resolve_packages_path())

In [ ]:
# Every mock file must exist now. The cross-covariance is not a mock: it is made
# from the mocks by create_cross_covariance.ipynb.
missing = [str(file) for key, file in MOCK.items() if key != "cross_cov" and not file.exists()]
if missing:
    print("mock files not found:", *missing, sep="\n  ")
else:
    print("all mock files found")

## 1. The galaxy x kappa twin

We evaluate the `galaxy_kappa` probe (`soliket.GalaxyKappaLikelihood`) on the mock at the fiducial
and replace the galaxy x kappa bandpowers by its binned theory. The twin keeps the windows and
covariance of the mock and only the cross-spectrum the likelihood uses (`use_spectra` in the
configuration). Parameter overrides (e.g. a shifted `b1`) imprint a non-fiducial twin.

In [ ]:
from cobaya.model import get_model
from cobaya.tools import resolve_packages_path

from soliket.sacc_tools import smooth_twin_sacc

GALAXY_KAPPA_LIKE = "soliket.GalaxyKappaLikelihood"


def smooth_galaxy_kappa(out_path, **param_overrides):
    """Write the smooth (theory) twin of the galaxy x kappa mock at the fiducial.

    `param_overrides` pins fiducial params (e.g. ``b1=1.2``) to imprint the twin
    at a non-fiducial point. Returns the binned galaxy x kappa theory.
    """
    info = iso_config.build_info("galaxy_kappa", level="mock")
    info["params"] = iso_config.fix_to_fiducial(info["params"])
    for name, value in param_overrides.items():
        info["params"][name] = {"value": value}
    info["packages_path"] = resolve_packages_path()

    model = get_model(info)
    model.loglikes({})  # evaluate at the imprint point
    like = model.likelihood[GALAXY_KAPPA_LIKE]
    biases = {name: info["params"][name]["value"] for name in ["b1", "s1"]}
    cl_gk = like._get_theory(**biases)  # binned galaxy x kappa theory

    galaxy, kappa = info["likelihood"][GALAXY_KAPPA_LIKE]["use_spectra"][0]
    src = sacc.Sacc.load_fits(like.datapath)  # reuse the mock's windows/cov
    smooth_twin_sacc(src, "cl_00", galaxy, kappa, cl_gk, out_path=out_path)
    return cl_gk


cl_gk = smooth_galaxy_kappa(TWIN["galaxy_kappa_data"])
print("wrote", TWIN["galaxy_kappa_data"], "->", len(cl_gk), "bins")

### Smooth galaxy x kappa twin vs. the mock

The twin lies as a clean curve through the mock it was carved from: the mock bandpowers scatter
around it within their errors. That is the chi-square = 0 statement, made visual.

We plot $\ell\, C_\ell^{g\kappa}$, which varies less steeply with $\ell$ than $C_\ell^{g\kappa}$.

In [ ]:
import matplotlib.pyplot as plt


def read_cl(path, t1, t2):
    """(ell, C_ell, err) for a cl_00 spectrum in a SACC file; err is None if no cov."""
    s = sacc.Sacc.load_fits(str(path))
    try:
        ell, cl, cov = s.get_ell_cl("cl_00", t1, t2, return_cov=True)
        return ell, cl, np.sqrt(np.diag(cov))
    except ValueError:  # e.g. files that carry only bandpowers
        ell, cl = s.get_ell_cl("cl_00", t1, t2, return_cov=False)
        return ell, cl, None


info = iso_config.build_info("galaxy_kappa", level="mock")
galaxy, kappa = info["likelihood"][GALAXY_KAPPA_LIKE]["use_spectra"][0]

fig, ax = plt.subplots(figsize=(7, 4.5))
ell, cl, err = read_cl(MOCK["galaxy_kappa_data"], galaxy, kappa)
ax.errorbar(ell, ell * cl, None if err is None else ell * err, fmt="o", ms=4, color="brown",
            label="mock")
ell, cl, _ = read_cl(TWIN["galaxy_kappa_data"], galaxy, kappa)
ax.plot(ell, ell * cl, "-", color="tab:blue", lw=2, label="smooth twin")
ax.set(xlabel=r"$\ell$", ylabel=r"$\ell\, C_\ell^{g\kappa}$", xscale="log",
       title=f"Galaxy x CMB lensing ({galaxy} x {kappa})")
ax.legend(); fig.tight_layout(); plt.show()

## 2. The CMB twins: MFLike and lensing, from one joint model

Both CMB twins are imprinted from the **joint** MFLike + lensing model,
`iso_config.build_info("multigaussian_mflike_lensing", level="mock")`, with every parameter fixed to
the fiducial. Imprinting from the model the joint analysis fits matters: its CAMB run uses the
precision settings of both probes, so twins built from standalone models would leave a small
residual chi-square in the joint fit. One CAMB run (at the joint accuracy, a few minutes) gives
both twins:

- lensing: the mock's bandpowers replaced by the binned $C_L^{\kappa\kappa}$, via `smooth_twin_sacc`;
- MFLike: CMB + foregrounds + systematics through MFLike's `get_modified_theory`, binned with its own
  bandpower windows, one `NuMap` tracer per `(frequency, spin)` channel, via `smooth_mflike_sacc`.

Only the data vectors are regenerated: the twins keep the mocks' covariances and windows (the
`mflike_cov_Bbl` and lensing keys point to the same files in both sections of `paths.yaml`).
The **imprint point is a knob**: by default the fiducial, but any param override (e.g. a shifted
`logA`) imprints both twins at a non-fiducial point, for parameter-recovery tests.

In [ ]:
from soliket.presets import resolve_aliases
from soliket.sacc_tools import smooth_mflike_sacc


def smooth_cmb(mflike_path, lensing_path, **param_overrides):
    """Write the smooth twins of the MFLike and CMB-lensing mocks, from one joint model.

    Both twins come from the same CAMB run, the one the joint fit recomputes.
    `param_overrides` pins params (e.g. ``logA=3.07``) to imprint both twins at a
    non-fiducial point. Returns the model's role aliases (``.mflike``, ``.lensing``).
    """
    info = iso_config.build_info("multigaussian_mflike_lensing", level="mock")
    # The cross-covariance does not affect the theory, and may not be built yet.
    info["likelihood"][iso_config.MULTIGAUSSIAN].pop("cross_cov_path", None)
    info["params"] = iso_config.fix_to_fiducial(info["params"])
    for name, value in param_overrides.items():
        info["params"][name] = {"value": value}
    info["packages_path"] = resolve_packages_path()

    model = get_model(info)
    model.loglikes({})  # evaluate at the imprint point
    roles = resolve_aliases(model)

    # Lensing: the binned C_L^kappakappa, with the mock's tracers, windows and cov.
    lensing = roles.lensing
    tracer1, tracer2 = lensing.tracer_comb
    src = sacc.Sacc.load_fits(lensing.datapath)
    clkk = lensing._get_theory()
    smooth_twin_sacc(src, "cl_00", tracer1, tracer2, clkk, out_path=lensing_path)

    # MFLike: CMB + foregrounds + systematics, binned with MFLike's windows.
    params = iso_config.fiducial_values(info["params"])  # numeric, overrides included
    dls = model.provider.get_Cl(ell_factor=True)
    fg_totals = roles.foreground.get_fg_totals()
    smooth_mflike_sacc(roles.mflike, dls, fg_totals, params, out_path=mflike_path)
    return roles


roles = smooth_cmb(TWIN["mflike_data"], TWIN["lensing_data"])
print("wrote", TWIN["mflike_data"], "and", TWIN["lensing_data"])

In [ ]:
# Knob - imprint both twins at a shifted amplitude, a recovery target for the logA
# chain of analyse_datasets.ipynb. They are written next to the fiducial twins with
# a prefix, and are not in paths.yaml: analyse_datasets.ipynb shows how to fit them.
SHIFT = {"logA": 3.07}
PREFIX = "logA3p07_"
SHIFTED = {key: TWIN[key].with_name(PREFIX + TWIN[key].name) for key in ["mflike_data", "lensing_data"]}
smooth_cmb(SHIFTED["mflike_data"], SHIFTED["lensing_data"], **SHIFT)
print("wrote", *SHIFTED.values())

### Smooth lensing twin vs. the mock

The `logA=3.07` twin is overlaid too: $C_L^{\kappa\kappa}$ scales roughly as $A_\mathrm{s}$, so a
shift of 0.02 in $\log(10^{10} A_\mathrm{s})$ raises it by about 2% at all $L$, the offset a
`logA`-recovery fit must find.

In [ ]:
tracer1, tracer2 = roles.lensing.tracer_comb
fig, ax = plt.subplots(figsize=(7, 4.5))
L, cl, err = read_cl(MOCK["lensing_data"], tracer1, tracer2)
ax.errorbar(L, cl, err, fmt="o", ms=4, color="brown", label="mock")
L, cl, _ = read_cl(TWIN["lensing_data"], tracer1, tracer2)
ax.plot(L, cl, "-", color="tab:blue", lw=2, label="smooth twin")
L, cl, _ = read_cl(SHIFTED["lensing_data"], tracer1, tracer2)
ax.plot(L, cl, "--", color="tab:orange", lw=1.5, label="smooth twin, logA = 3.07")
ax.set(xlabel="$L$", ylabel=r"$C_L^{\kappa\kappa}$", title="CMB lensing", xscale="log")
ax.legend(); fig.tight_layout(); plt.show()

### Smooth CMB twin vs. the mock

For the first temperature channel of the twin, the mock bandpowers scatter around the smooth curve —
acoustic peaks at low ℓ, the foreground-dominated rise at high ℓ. The `logA=3.07` twin sits about
2% higher where the CMB dominates. (The MFLike covariance lives in the separate `cov_Bbl` file, so
we overlay bandpowers only.)

In [ ]:
twin = sacc.Sacc.load_fits(str(TWIN["mflike_data"]))
channel, _ = twin.get_tracer_combinations("cl_00")[0]  # first TT auto-spectrum
fig, ax = plt.subplots(figsize=(7, 4.5))
ell, cl, _ = read_cl(MOCK["mflike_data"], channel, channel)
ax.plot(ell, cl, ".", ms=4, alpha=0.45, color="grey", label="mock")
ell, cl, _ = read_cl(TWIN["mflike_data"], channel, channel)
ax.plot(ell, cl, "-", color="tab:red", lw=1.5, label="smooth twin")
ell, cl, _ = read_cl(SHIFTED["mflike_data"], channel, channel)
ax.plot(ell, cl, "--", color="tab:orange", lw=1.2, label="smooth twin, logA = 3.07")
ax.set(xlabel=r"$\ell$", ylabel=r"$D_\ell^{TT}\ [\mu K^2]$",
       title=f"MFLike TT {channel}: smooth twin vs mock")
ax.legend(); fig.tight_layout(); plt.show()

## Recap

The `twin` files of `paths.yaml` now exist:

| Key of `paths.yaml` | Built from | Consumed by |
| --- | --- | --- |
| `galaxy_kappa_data` | `smooth_twin_sacc` (galaxy x kappa theory) | the `galaxy_kappa` probe |
| `lensing_data` | `smooth_twin_sacc` (lensing theory, joint model) | the lensing component |
| `mflike_data` | `smooth_mflike_sacc` (CMB + fg theory, same joint model) | the MFLike component |

Everything else a twin needs (covariances, windows, lensing corrections) is the mock's, through
`paths.yaml`. All twins are built at the fiducial the analysis is checked at, so the chi-square of
each twin is 0 there by construction. The `logA3p07_` twins of the knob sit next to them, outside
`paths.yaml`.

Next: [`create_cross_covariance.ipynb`](create_cross_covariance.ipynb) builds the cross-covariance
between the MFLike and lensing data vectors; then [`analyse_datasets.ipynb`](analyse_datasets.ipynb)
runs the joint fit.